# Credit Risk Scoring Analysis

Análisis exploratorio, entrenamiento y explicabilidad de modelo XGBoost para predecir impago de crédito.

- AUC objetivo: 77.6%
- Dataset: 30,000 clientes (OpenML CCF Taiwan)
- Interpretabilidad: SHAP TreeExplainer

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, classification_report
from xgboost import XGBClassifier
import shap
import joblib

sns.set_style('darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)

RANDOM_STATE = 42

## 1. Carga de datos

In [ ]:
# Descargar dataset desde OpenML
print('Descargando dataset default-of-credit-card-clients...')
dataset = fetch_openml(
    name='default-of-credit-card-clients',
    version=1,
    as_frame=True,
    parser='auto'
)
frame = dataset.frame.copy()

print(f'Dataset shape: {frame.shape}')
print(f'Columns: {frame.columns.tolist()}')
print(f'\nPrimeras filas:')
frame.head()

In [ ]:
# Preparar X, y
target_col = dataset.target_names[0]
y = frame[target_col].astype(int)
X = frame.drop(columns=[target_col])

# Renombrar columnas si es necesario (OpenML usa x1..x23)
mapping = {
    'x1': 'LIMIT_BAL', 'x2': 'SEX', 'x3': 'EDUCATION', 'x4': 'MARRIAGE',
    'x5': 'AGE', 'x6': 'PAY_0', 'x7': 'PAY_2', 'x8': 'PAY_3',
    'x9': 'PAY_4', 'x10': 'PAY_5', 'x11': 'PAY_6',
    'x12': 'BILL_AMT1', 'x13': 'BILL_AMT2', 'x14': 'BILL_AMT3',
    'x15': 'BILL_AMT4', 'x16': 'BILL_AMT5', 'x17': 'BILL_AMT6',
    'x18': 'PAY_AMT1', 'x19': 'PAY_AMT2', 'x20': 'PAY_AMT3',
    'x21': 'PAY_AMT4', 'x22': 'PAY_AMT5', 'x23': 'PAY_AMT6'
}
rename_dict = {col: mapping.get(col, col) for col in X.columns}
X = X.rename(columns=rename_dict)

print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}')
print(f'\nClase balance:')
print(y.value_counts())
print(f'\nPorcentaje default: {y.mean()*100:.2f}%')

## 2. EDA (Exploratory Data Analysis)

In [ ]:
# Estadísticas descriptivas
X.describe()

In [ ]:
# Correlación con target
X_with_target = X.copy()
X_with_target['default'] = y

correlations = X_with_target.corr()['default'].sort_values(ascending=False)
print('Top correlaciones con default:')
print(correlations.head(10))

fig, ax = plt.subplots(figsize=(10, 6))
correlations[1:11].plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Top 10 Features vs Default (Correlación)', fontsize=12, fontweight='bold')
ax.set_xlabel('Correlación')
plt.tight_layout()
plt.show()

## 3. Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

print(f'Train shape: {X_train.shape}')
print(f'Test shape: {X_test.shape}')
print(f'\nTrain default rate: {y_train.mean()*100:.2f}%')
print(f'Test default rate: {y_test.mean()*100:.2f}%')

## 4. Entrenamiento del modelo

In [ ]:
# Definir pipeline
categorical_features = ['SEX', 'EDUCATION', 'MARRIAGE']
numeric_features = [col for col in X_train.columns if col not in categorical_features]

preprocessor = ColumnTransformer(
    transformers=[
        ('categorical', OneHotEncoder(handle_unknown='ignore'), categorical_features),
        ('numeric', 'passthrough', numeric_features)
    ]
)

classifier = XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    n_estimators=500,
    learning_rate=0.05,
    max_depth=4,
    min_child_weight=4,
    subsample=0.85,
    colsample_bytree=0.8,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1
)

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', classifier)
])

print('Entrenando modelo...')
pipeline.fit(X_train, y_train)
print('✓ Modelo entrenado')

## 5. Evaluación

In [ ]:
# Predicciones
y_pred_train = pipeline.predict(X_train)
y_pred_test = pipeline.predict(X_test)
y_proba_train = pipeline.predict_proba(X_train)[:, 1]
y_proba_test = pipeline.predict_proba(X_test)[:, 1]

# AUC
auc_train = roc_auc_score(y_train, y_proba_train)
auc_test = roc_auc_score(y_test, y_proba_test)

print(f'AUC Train: {auc_train:.4f}')
print(f'AUC Test:  {auc_test:.4f}')
print(f'\nTest Set Classification Report:')
print(classification_report(y_test, y_pred_test, target_names=['No Default', 'Default']))

In [ ]:
# ROC Curve
fpr, tpr, thresholds = roc_curve(y_test, y_proba_test)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(fpr, tpr, linewidth=2.5, label=f'ROC (AUC = {auc_test:.3f})')
ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Classifier')
ax.set_xlabel('False Positive Rate', fontsize=11)
ax.set_ylabel('True Positive Rate', fontsize=11)
ax.set_title('ROC Curve - Test Set', fontsize=12, fontweight='bold')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_test)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax, cbar=False,
            xticklabels=['No Default', 'Default'],
            yticklabels=['No Default', 'Default'])
ax.set_ylabel('True Label', fontsize=11)
ax.set_xlabel('Predicted Label', fontsize=11)
ax.set_title('Confusion Matrix - Test Set', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Importancia de Features (SHAP)

In [ ]:
# Calcular SHAP values
print('Calculando SHAP values (puede tomar 1-2 minutos)...')
X_test_transformed = pipeline.named_steps['preprocessor'].transform(X_test[:1000])  # Sample para rapidez
model = pipeline.named_steps['classifier']

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test_transformed)

# Si es lista, tomar clase 1 (default)
if isinstance(shap_values, list):
    shap_values = shap_values[1]

print('✓ SHAP values calculados')

In [ ]:
# Feature names después de preprocesamiento
feature_names = pipeline.named_steps['preprocessor'].get_feature_names_out().tolist()

# Mean absolute SHAP
mean_abs_shap = np.abs(shap_values).mean(axis=0)
top_idx = np.argsort(mean_abs_shap)[::-1][:15]

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(range(len(top_idx)), mean_abs_shap[top_idx], color='steelblue')
ax.set_yticks(range(len(top_idx)))
ax.set_yticklabels([feature_names[i] for i in top_idx])
ax.set_xlabel('Mean |SHAP value|', fontsize=11)
ax.set_title('Top 15 Features by Mean |SHAP|', fontsize=12, fontweight='bold')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# SHAP Summary (waterfall para caso individual)
sample_idx = 0
print(f'\nExplicación SHAP para muestra {sample_idx}:')
print(f'Probabilidad de default: {y_proba_test[sample_idx]:.4f}')
print(f'Etiqueta real: {y_test.iloc[sample_idx]}')

# Mostrar top contribuyentes
sample_shap = shap_values[sample_idx]
top_contrib_idx = np.argsort(np.abs(sample_shap))[::-1][:8]

print(f'\nTop contribuyentes (SHAP):')
for i, idx in enumerate(top_contrib_idx, 1):
    direction = 'aumenta riesgo' if sample_shap[idx] > 0 else 'disminuye riesgo'
    print(f'{i}. {feature_names[idx]:30s} SHAP={sample_shap[idx]:+.4f} ({direction})')

## 7. Guardar Modelo (Opcional)

In [ ]:
# El modelo ya está guardado en src/train.py, pero puedes hacer backup aquí
# model_path = Path('models/xgb_credit_pipeline.joblib')
# model_path.parent.mkdir(exist_ok=True)
# joblib.dump(pipeline, model_path)
# print(f'Modelo guardado en {model_path}')

print(f'\n✓ Análisis completado')
print(f'\nResumen:')
print(f'  - AUC Test: {auc_test:.4f}')
print(f'  - Features: {X.shape[1]}')
print(f'  - Muestras de entrenamiento: {X_train.shape[0]}')
print(f'  - Muestras de test: {X_test.shape[0]}')